## Заменяем примеры из GOLD в результатах модели

> NOTE: тут перевод не верифицированный весь получаем, а верифицированы только элементы ГОЛДа

- Берем все элементы датасетов (dev,train,test), переведенные и с перенесенной разметкой с помощью модели.
- добавляем поле is_gold поле для элементов
- Для тех, которые есть в gold - мы ставим 1, для тех которых нет в голде - ставим 0. 
- Но для is_gold=1 элементов ЗАМЕНЯЕМ элемент на валидированный перевод и валидированную разметку из GOLD.


#### Алгоритм

1. формируем индекс для быстрого доступа по id к данным GOLD
2. проходимся по файлу датасета и :
    - для тех id, которые из ГОЛД - добавляем новое поле 'is_gold' = 1 и заменяем элемент на этот элемент из GOLD
    - для остальных - 'is_gold' = 0


<!-- ##### должно получиться 25363 итого в сумме -->


In [44]:
import json
from tqdm import tqdm
import logging
from datasets import Dataset
import datetime


Настройка логгера, для вывода подробной информации работа скрипта в файл `log.log`

При этом основной вывод прогрессбара от `tqdm` будет осуществляться в `Out` ячейку

In [45]:
# Настройка логгера для записи в файл
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

# Создание обработчика для записи в файл
file_handler = logging.FileHandler('log.log')
file_handler.setLevel(logging.DEBUG)

# Создание форматтера для сообщений
FORMAT = "%(asctime)s %(levelname)s [%(process)s]: %(name)s: %(message)s"
formatter = logging.Formatter(FORMAT)
file_handler.setFormatter(formatter)

# Добавление обработчика к логгеру
logger.addHandler(file_handler)

In [46]:
import os

def create_dir_if_not_exists(directory: str) -> str:
    """Создаёт директорию, если она не существует.
    
    Args:
        directory (str): Путь к директории.
    
    Returns:
        str: Абсолютный путь к созданной/существующей директории.
    """
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Директория создана: {directory}")
    else:
        print(f"Директория уже существует: {directory}")
    return os.path.abspath(directory)

Задаем переменные

In [47]:
gold_file = "/mnt/tank/rudeft_dmitry/dpopov/rudeft_gold/gold.json"

dataset_dir = "/mnt/tank/rudeft_dmitry/dpopov/rudeft_after_qwen"
# dataset_dir = "/mnt/tank/rudeft_dmitry/dpopov/rudeft_after_deepseek_chat"

new_files_dir = f"{dataset_dir}_replaced_gold"

In [48]:
# Создаём директорию (если её нет)
abs_path = create_dir_if_not_exists(new_files_dir)
print(f"Абсолютный путь: {abs_path}")

Директория создана: /mnt/tank/rudeft_dmitry/dpopov/rudeft_after_qwen_replaced_gold
Абсолютный путь: /mnt/tank/rudeft_dmitry/dpopov/rudeft_after_qwen_replaced_gold


Исходные файлы, полученные после пайплайна перевода и переноса разметки от LLM

In [49]:
dev_file = f"{dataset_dir}/dev.json"
test_file = f"{dataset_dir}/test.json"
train_file = f"{dataset_dir}/train.json"

Новые файлы после обработки

In [50]:
new_dev_file = f"{new_files_dir}/dev.json"
new_test_file = f"{new_files_dir}/test.json"
new_train_file = f"{new_files_dir}/train.json"

Главная функция

In [ ]:
def replace_gold_elems_in_dataset(dataset_file, gold_file, new_file):
    """ Функция добавления GOLD примеров и поля is_gold в датасет.

    Создаем датасет, в котором:
    - Для тех элементов, которых нет в голде 
        - ставим is_gold=0
    - Для тех элементов, которые есть в gold:
        - заменяем элемент на полностью элемент из GOLD
        - ставим is_gold=1

    Все файлы в HF формате.

    :param dataset_file: путь к файлу датасета (после LLM перевода и переноса разметки).
    :param gold_file: путь к файлу GOLD датасета.
    :param new_file: путь к новому итоговому файлу датасета.

    """
    # считываем данные из hf датасета   
    orig_data = Dataset.from_json(dataset_file)       
    orig_data = orig_data.to_list()
    hf_gold = Dataset.from_json(gold_file)  
    hf_gold = hf_gold.to_list()


# 1. формируем индекс для быстрого доступа по id к данным GOLD
    id2gold = {}
    for elem in tqdm(hf_gold, desc="Create index id2gold"):
        id2gold[elem["id"]] = elem


# 2. проходимся по файлу датасета и :
#     - для тех id, которые из ГОЛД - добавляем новое поле 'is_gold' = 1 и заменяем элемент на этот элемент из GOLD
#     - для остальных - 'is_gold' = 0
    logging.info(f"Processing '{dataset_file}'...")
    for elem in tqdm(orig_data, desc="Adding is_gold field and replacing gold elems"):
        if elem['id'] in id2gold:
            gold_elem = id2gold[elem['id']]
            logging.info(f"Updating {elem['id']}:")
            logging.info(f"elem before: {elem}")
            elem.update(gold_elem)
            elem['is_gold'] = 1
            logging.info(f"elem after : {elem}")
        else:
            elem['is_gold'] = 0
    logging.info(f"Processed '{dataset_file}'.")

    
    # Записываем данные в HF датасет
    orig_data = Dataset.from_list(orig_data)
    orig_data.to_json(new_file, force_ascii=False)

    logging.info(f"Length new DATASET: {len(orig_data)}")
    logging.info(f"Info new DATASET: {orig_data}")

In [52]:
replace_gold_elems_in_dataset(
    dataset_file=dev_file,
    gold_file=gold_file,
    new_file=new_dev_file
    )

Generating train split: 1179 examples [00:00, 53471.36 examples/s]


Creating json from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 47.37ba/s]


In [53]:
replace_gold_elems_in_dataset(
    dataset_file=test_file,
    gold_file=gold_file,
    new_file=new_test_file
    )

Generating train split: 1189 examples [00:00, 83887.49 examples/s]
Creating json from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 46.56ba/s]


In [54]:
replace_gold_elems_in_dataset(
    dataset_file=train_file,
    gold_file=gold_file,
    new_file=new_train_file
    )

Generating train split: 24184 examples [00:00, 31661.29 examples/s]
Creating json from Arrow format: 100%|██████████| 25/25 [00:00<00:00, 28.96ba/s]
